In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import sqlite3
import pandas as pd

In [2]:
conn = sqlite3.connect("../../db/dev.sqlite")

In [3]:
df = pd.read_csv("../raw-data/worldbank_data.csv", encoding="latin1", engine="python", na_values = '..')

In [4]:
df.describe()
metadata = pd.read_csv('../raw-data/worldbank_metadata.csv', encoding = 'latin1')

In [5]:
df_lat_long = pd.read_csv('../raw-data/longitude-latitude.csv')
df_lat_long=df_lat_long[['FIFA', 'Latitude', 'Longitude', 'ISO-ALPHA-3','Country']]
df_lat_long.columns=df_lat_long.columns.str.lower()
df_lat_long['country'] = df_lat_long['country'].replace({
    'Jersey': 'Channel Islands'
})

In [6]:
years_cols = [col for col in df.columns if col[:4].isdigit()]

df_long = df.melt(
    id_vars=['Country Name', 'Country Code', 'Series Name', 'Series Code'],
    value_vars=years_cols,
    var_name = 'year',
    value_name = 'value'
)

In [7]:
df_long["year"] = df_long["year"].str.extract(r"(\d{4})")

In [8]:
df_wide = df_long.pivot_table(
    index = ['Country Name', 'Country Code', 'year'],
    columns = 'Series Name',
    values = 'value'
).reset_index()

In [ ]:
df_population = pd.read_csv('../raw-data/world-population-data.csv', encoding="latin1", engine="python", na_values = '..')

population_years_cols = [col for col in df_population.columns if col[:4].isdigit()]

df_population = df_population.melt(
    id_vars=['Country Name', 'Country Code', 'Series Name', 'Series Code'],
    value_vars=population_years_cols,
    var_name = 'year',
    value_name = 'value'
)
df_population["year"] = df_population["year"].str.extract(r"(\d{4})")

df_population = df_population.pivot_table(
    index = ['Country Name', 'Country Code', 'year'],
    columns = 'Series Name',
    values = 'value'
).reset_index()

df_population  = df_population[['Population growth (annual %)', 'Population, total', 'Country Code']]
df_wide = df_wide.merge(df_population, left_on='Country Code', right_on='Country Code', how='left')

Series Name,Country Name_x,Country Code,year_x,GDP growth (annual %),"Inflation, consumer prices (annual %)","Life expectancy at birth, total (years)",Poverty gap at $3.00 a day (2021 PPP) (%),Country Name_y,year_y,Population growth (annual %)_x,"Population, total_x",Country Name,year,Population growth (annual %)_y,"Population, total_y"
0,Afghanistan,AFG,1960,NaN,NaN,32.799,NaN,Afghanistan,1990,1.434588,12045660.0,Afghanistan,1990,1.434588,12045660.0
1,Afghanistan,AFG,1960,NaN,NaN,32.799,NaN,Afghanistan,1990,1.434588,12045660.0,Afghanistan,1991,1.591326,12238879.0
2,Afghanistan,AFG,1960,NaN,NaN,32.799,NaN,Afghanistan,1990,1.434588,12045660.0,Afghanistan,1992,8.156419,13278974.0
3,Afghanistan,AFG,1960,NaN,NaN,32.799,NaN,Afghanistan,1990,1.434588,12045660.0,Afghanistan,1993,11.807259,14943172.0
4,Afghanistan,AFG,1960,NaN,NaN,32.799,NaN,Afghanistan,1990,1.434588,12045660.0,Afghanistan,1994,8.388730,16250794.0


In [ ]:
df_wide = df_wide.rename(columns={
    'Country Name' : 'country',
    'Country Code' : 'country_code',
    'GDP growth (annual %)' : 'gdp_growth',
    'Inflation, consumer prices (annual %)': 'inflation',
    'Life expectancy at birth, total (years)': 'life_expectancy',
    'Poverty gap at $3.00 a day (2021 PPP) (%)' : 'poverty'
    'population' : ''
})

In [ ]:
df_wide.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17178 entries, 0 to 17177
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   country          17178 non-null  object 
 1   country_code     17178 non-null  object 
 2   year             17178 non-null  object 
 3   gdp_growth       14133 non-null  float64
 4   inflation        11295 non-null  float64
 5   life_expectancy  16926 non-null  float64
 6   poverty          2970 non-null   float64
dtypes: float64(4), object(3)
memory usage: 939.6+ KB


In [ ]:
df_wide['year'] = df_wide['year'].astype(int)

In [ ]:
df_wide = df_wide[df_wide['year'] >= 1990]

In [ ]:
df_wide['country'].unique()

array(['Afghanistan', 'Africa Eastern and Southern',
       'Africa Western and Central', 'Albania', 'Algeria',
       'American Samoa', 'Andorra', 'Angola', 'Antigua and Barbuda',
       'Arab World', 'Argentina', 'Armenia', 'Aruba', 'Australia',
       'Austria', 'Azerbaijan', 'Bahamas, The', 'Bahrain', 'Bangladesh',
       'Barbados', 'Belarus', 'Belgium', 'Belize', 'Benin', 'Bermuda',
       'Bhutan', 'Bolivia', 'Bosnia and Herzegovina', 'Botswana',
       'Brazil', 'British Virgin Islands', 'Brunei Darussalam',
       'Bulgaria', 'Burkina Faso', 'Burundi', 'Cabo Verde', 'Cambodia',
       'Cameroon', 'Canada', 'Caribbean small states', 'Cayman Islands',
       'Central African Republic', 'Central Europe and the Baltics',
       'Chad', 'Channel Islands', 'Chile', 'China', 'Colombia', 'Comoros',
       'Congo, Dem. Rep.', 'Congo, Rep.', 'Costa Rica', "Cote d'Ivoire",
       'Croatia', 'Cuba', 'Curacao', 'Cyprus', 'Czechia', 'Denmark',
       'Djibouti', 'Dominica', 'Dominican Repub

In [ ]:
not_countries = [
"Africa Eastern and Southern",
"Africa Western and Central",
"Arab World",
"Caribbean small states",
"Central Europe and the Baltics",
"Early-demographic dividend",
"East Asia & Pacific",
"East Asia & Pacific (IDA & IBRD countries)",
"East Asia & Pacific (excluding high income)",
"Euro area",
"Europe & Central Asia",
"Europe & Central Asia (IDA & IBRD countries)",
"Europe & Central Asia (excluding high income)",
"European Union",
"Fragile and conflict affected situations",
"Heavily indebted poor countries (HIPC)",
"High income",
"IBRD only",
"IDA & IBRD total",
"IDA blend",
"IDA only",
"IDA total",
"Late-demographic dividend",
"Latin America & Caribbean",
"Latin America & Caribbean (excluding high income)",
"Latin America & the Caribbean (IDA & IBRD countries)",
"Least developed countries: UN classification",
"Low & middle income",
"Low income",
"Lower middle income",
"Middle East, North Africa, Afghanistan & Pakistan",
"Middle East, North Africa, Afghanistan & Pakistan (IDA & IBRD)",
"Middle East, North Africa, Afghanistan & Pakistan (excluding high income)",
"Middle income",
"North America",
"OECD members",
"Other small states",
"Pacific island small states",
"Post-demographic dividend",
"Pre-demographic dividend",
"Small states",
"South Asia",
"South Asia (IDA & IBRD)",
"Sub-Saharan Africa",
"Sub-Saharan Africa (IDA & IBRD countries)",
"Sub-Saharan Africa (excluding high income)",
"Upper middle income",
"World"
]

df_wide = df_wide[~df_wide['country'].isin(not_countries)]

In [ ]:
df_wide['country'].nunique()

217

In [ ]:
metadata = pd.read_csv('../raw-data/worldbank_metadata.csv', encoding = 'latin1')

In [ ]:
df_wide = df_wide.merge(metadata, left_on = 'country_code', right_on='Country Name', how = 'left')

In [ ]:
iso_lookup = df_lat_long[['iso-alpha-3', 'latitude', 'longitude']].drop_duplicates(subset=['iso-alpha-3'])
country_lookup = df_lat_long[['country', 'latitude', 'longitude']].drop_duplicates(subset=['country'])


df_merged = df_wide.merge(
    iso_lookup,
    left_on='country_code',
    right_on='iso-alpha-3',
    how='left'
)

df_merged = df_merged.drop(columns=['iso-alpha-3'])

missing_mask = df_merged['latitude'].isna()

fallback = df_merged.loc[missing_mask].drop(columns=['latitude','longitude']).merge(
    country_lookup,
    left_on='country',
    right_on='country',
    how='left'
)

fallback = fallback.drop(columns=['country'])

df_merged.loc[missing_mask, 'latitude'] = fallback['latitude'].values
df_merged.loc[missing_mask, 'longitude'] = fallback['longitude'].values

df_wide = df_merged

In [ ]:
df_wide = df_wide.rename(columns= {
    'Series Code': 'region',
    'Series Name': 'income'
})

In [ ]:
df_wide = df_wide[['country', 'country_code', 'region', 'income', 'year', 'gdp_growth', 'inflation', 'life_expectancy', 'poverty', 'latitude', 'longitude']]

In [ ]:
df_wide.to_csv("../raw-data/economic_data_clean.csv", index = False)

In [ ]:
df_latest = df_wide[df_wide['year']==2022]

In [ ]:
df_latest.to_csv('../raw-data/economic_latest.csv', index = False)

In [ ]:
df_wide.to_sql(
    "economic_history",
    conn,
    if_exists="replace",
    index=False
)

7578

In [ ]:
df_latest.to_sql(
    "economic_latest",
    conn,
    if_exists="replace",
    index=False
)

217

In [ ]:
conn.close()